# Perturbation Prediction Benchmark — Tutorial

This notebook walks you through evaluating perturbation prediction models
on your own `.h5ad` Perturb-seq dataset using the modular evaluation toolkit.

## Supported Models

| Model | Approach | GPU Required | Reference |
|-------|----------|--------------|-----------|
| **GEARS** | Graph Auto-Regressive | Yes | [Roohani et al. 2023](https://github.com/snap-stanford/GEARS) |
| **scGPT** | Transformer (pre-trained) | Yes | [Cui et al. 2024](https://github.com/bowang-lab/scGPT) |
| **STATE** | Zero-shot HVG Transfer | Yes | [Arc Institute](https://huggingface.co/arcinstitute/ST-HVG-Replogle) |
| **Cell2Sentence** | LLM (Gemma-2-2B, 4-bit) | Yes (CUDA) | [Van Dijk Lab](https://github.com/vandijklab/cell2sentence) |
| **CPA** | Conditional VAE | Optional | [Lotfollahi et al. 2023](https://github.com/theislab/cpa) |

## Evaluation Tiers

All models are scored on **8 metrics** across 3 tiers:

| Tier | Metric | Direction |
|------|--------|-----------|
| T1 | Centroid Accuracy (CA) | ↑ higher is better |
| T1 | Profile Distance Score (PDS) | ↓ lower is better |
| T1 | Systema Pearson Delta | ↑ higher is better |
| T2 | Directional Accuracy | ↑ higher is better |
| T2 | Pearson Delta Top-K | ↑ higher is better |
| T2 | Jaccard DE Top-K | ↑ higher is better |
| T3 | Energy Distance | ↓ lower is better |
| T3 | MMD (RBF kernel) | ↓ lower is better |

## Step 1: Environment Setup

**Google Colab users**: Make sure you select a GPU runtime (`Runtime → Change runtime type → T4 GPU`).

Each model installs its own dependencies automatically on first run, so you only need the base packages below.

In [ ]:
# Clone the repository (uncomment if not already cloned)
# !git clone https://github.com/kagtgi/PerturbationBenchmarkTool.git
# %cd PerturbationBenchmarkTool

# Install base dependencies
!pip install -q anndata scanpy scipy pandas matplotlib torch

## Step 2: Download Dataset

You need a `.h5ad` Perturb-seq file. The default dataset is the K562 CRISPR screen.

If you already have the file, skip this cell and just update the path in Step 3.

In [ ]:
import os

# Download K562 dataset if not present
DATA_FILE = "K562.h5ad"
if not os.path.exists(DATA_FILE):
    print("Downloading K562 dataset...")
    !wget -q https://huggingface.co/datasets/lep1106/perturbation-data-K562/resolve/main/K562.h5ad
    print(f"Downloaded: {DATA_FILE}")
else:
    print(f"Dataset already exists: {DATA_FILE}")

## Step 3: Configure

Set the path to your `.h5ad` dataset and choose evaluation parameters.
All parameters have sensible defaults — you only need to change `DATA_PATH` if your file is named differently.

In [ ]:
from eval import config

# === EDIT THESE AS NEEDED ===
config.DATA_PATH = "K562.h5ad"       # Path to your .h5ad Perturb-seq file
config.DEVICE = "cuda"               # "cuda" or "cpu"
config.RANDOM_SEED = 42              # For reproducibility
config.CTRL_LABEL = "non-targeting"  # Control cell label in obs[PERT_COL]
config.PERT_COL = "gene"             # Column in obs with perturbation labels
config.OUTPUT_DIR = "results"        # Where to save outputs

# Display active configuration
print("Active configuration:")
print(f"  Data path:    {config.DATA_PATH}")
print(f"  Device:       {config.DEVICE}")
print(f"  Random seed:  {config.RANDOM_SEED}")
print(f"  Control label: {config.CTRL_LABEL}")
print(f"  Pert column:  {config.PERT_COL}")
print(f"  Subsample:    {config.SUBSAMPLE_FRAC*100:.0f}%")
print(f"  Top-K DE:     {config.TOP_K_DE}")
print(f"  Output dir:   {config.OUTPUT_DIR}")

## Step 4: Load & Subsample Dataset

The framework loads the `.h5ad` file and performs **stratified subsampling** to ensure fair comparison.
Every model evaluates on the **exact same cells** — this is critical for a fair benchmark.

Subsampling keeps 20% of cells per perturbation group (configurable), with a minimum of 5 cells per group.

In [ ]:
from eval.dataset import load_h5ad, ensure_raw_counts
from eval.sampling import stratified_subsample

# Load dataset
adata = load_h5ad(config.DATA_PATH)
adata = ensure_raw_counts(adata)
print(f"Full dataset:   {adata.shape[0]} cells × {adata.shape[1]} genes")

# Stratified subsample (identical across all models)
adata_sub = stratified_subsample(adata)
n_perts = adata_sub.obs[config.PERT_COL].nunique() - 1  # exclude control
n_ctrl = (adata_sub.obs[config.PERT_COL] == config.CTRL_LABEL).sum()
print(f"Subsampled:     {adata_sub.shape[0]} cells × {adata_sub.shape[1]} genes")
print(f"Perturbations:  {n_perts}")
print(f"Control cells:  {n_ctrl}")

## Step 5: Run Model Evaluations

### Option A: Run one model at a time (recommended for Colab)

Each model auto-installs its own dependencies on first run. On Colab, it is **recommended to restart the runtime** between models to free GPU memory and avoid dependency conflicts.

**Workflow**: Run one model → save results → restart runtime → re-run Steps 1-4 → run next model.

Change `MODEL_NAME` below to evaluate different models:
- `"gears"` — GEARS (graph-based, Norman pretrained)
- `"scgpt"` — scGPT (transformer, scGPT_human checkpoint)
- `"state"` — STATE (zero-shot, Arc Institute)
- `"cell2sentence"` — Cell2Sentence (LLM, Gemma-2-2B 4-bit)
- `"cpa"` — CPA (conditional VAE)

In [ ]:
from eval.models import run_model_eval
import json, os

# ============================================================
# Choose one model at a time
MODEL_NAME = "gears"  # Change this for each run
# ============================================================

cfg = {
    "DATA_PATH": config.DATA_PATH,
    "RANDOM_SEED": config.RANDOM_SEED,
    "DEVICE": config.DEVICE,
    "CTRL_LABEL": config.CTRL_LABEL,
    "PERT_COL": config.PERT_COL,
    "TOP_K_DE": config.TOP_K_DE,
    "MIN_CELLS_PER_PERT": config.MIN_CELLS_PER_PERT,
    "MAX_T3_CELLS": config.MAX_T3_CELLS,
    "OUTPUT_DIR": config.OUTPUT_DIR,
}

result = run_model_eval(MODEL_NAME, adata_sub.copy(), cfg)

# Display results
print(f"\nModel: {result['model']}")
print(f"Perturbations evaluated: {len(result['pert_names'])}")
print(f"Runtime: {result['runtime_seconds']:.1f}s")
print(f"\n{'Metric':<40} {'Value':>10}")
print("-" * 52)
for k, v in result['metrics'].items():
    print(f"  {k:<38} {v:>10.4f}")

# Save result
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
out_path = f"{config.OUTPUT_DIR}/{MODEL_NAME}_results.json"
with open(out_path, "w") as f:
    json.dump(result, f, indent=2, default=str)
print(f"\nSaved to {out_path}")

### Option B: Run all models via eval_runner

This runs all models sequentially in a single process. Best used on a machine with sufficient GPU memory (16+ GB).

**Note**: On Colab, prefer Option A with runtime restarts between models to avoid OOM errors and dependency conflicts.

In [ ]:
from eval.eval_runner import run

# Run all models (or specify a subset)
results = run(
    data_path=config.DATA_PATH,
    models=["gears", "scgpt", "state", "cell2sentence", "cpa"],
    device=config.DEVICE,
    seed=config.RANDOM_SEED,
    output_dir=config.OUTPUT_DIR,
)

# Quick summary
for r in results:
    status = "OK" if "error" not in r else f"FAILED: {r['error']}"
    print(f"  {r['model']:<20} {status}")

## Step 6: Collect & Compare Results

After running all models (across multiple sessions if needed), aggregate results into a comparison table.
The collector reads all `*_results.json` files from the output directory.

In [ ]:
from eval.collect_results import collect, pretty_table

df = collect(config.OUTPUT_DIR)

if df.empty:
    print("No results found yet. Run at least one model first (Step 5).")
else:
    print(pretty_table(df))

In [ ]:
# View as DataFrame for further analysis
if not df.empty:
    display(df)

In [ ]:
# Export to CSV
if not df.empty:
    csv_path = f"{config.OUTPUT_DIR}/final_comparison.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved to {csv_path}")

## Step 7: Interpreting Results

### Tier 1 — Whole-Profile Identity

These metrics evaluate whether the model captures the overall expression profile of the perturbation.

- **Centroid Accuracy (CA)** ↑: For each perturbation, is the predicted centroid closer to the true centroid than to any other perturbation's centroid? Based on rank ordering of Euclidean distances.
- **Profile Distance Score (PDS)** ↓: Normalized distance ranking — a score of 0 means the prediction is the closest match, 1 means it's the farthest.
- **Systema Pearson Delta** ↑: Pearson correlation between predicted and true expression changes (vs. control), averaged across all perturbations.

### Tier 2 — DE Gene Accuracy

These metrics focus on the top differentially expressed genes, which are biologically most informative.

- **Directional Accuracy** ↑: Among DE genes with |Δexpression| > threshold, what fraction has the correct direction (up vs. down)?
- **Pearson Delta Top-K** ↑: Pearson correlation of expression changes restricted to the top-K DE genes.
- **Jaccard Top-K** ↑: Overlap between the model's top-K predicted DE genes and the true top-K DE genes.

### Tier 3 — Distributional Fidelity

These metrics assess whether the model captures cell-to-cell variability, not just the mean.

- **Energy Distance** ↓: A statistical distance between the predicted and true cell distributions. Uses pairwise distances with off-diagonal correction.
- **MMD (RBF kernel)** ↓: Maximum Mean Discrepancy with a Gaussian kernel. Bandwidth is set via the median heuristic on true cell distances.

### Tips for Interpretation

- Models may excel at different tiers — a model with good T1 scores but poor T3 may predict means well but not distributions.
- **GEARS** produces point predictions (one centroid per perturbation), so its T3 scores use a single-point "distribution".
- **CPA** works in a log1p-normalized subspace with non-zero-variance genes, so its metrics are computed in that space.
- Gene vocabulary overlap matters for transfer-based models (GEARS, scGPT). Check `n_overlap_genes` in results.

## CLI Alternative

You can also run evaluations from the command line:

```bash
# Run specific models
python -m eval --data K562.h5ad --models gears scgpt --device cuda

# Run all models
python -m eval --data K562.h5ad --models all

# Collect and display results
python -m eval.collect_results --results-dir results/
```